<a href="https://colab.research.google.com/github/abbas-707/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas-707/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

##Git Clone

In [ ]:
!git clone https://github.com/abbas-707/FlyRank-Internship.git
%cd FlyRank-Internship

Cloning into 'FlyRank-Internship'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 133 (delta 46), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.85 MiB | 9.98 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/FlyRank-Internship/FlyRank-Internship


##0. DuckDB connection to the Hugging Face data

In [ ]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

# Quick test: list what's in the release
con.sql("""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
LIMIT 5
""").show()

┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ false          │ no_sea

## 1. Unit of analysis + time window

**One row = one content page, on one day, for one client.** Specifically, one row of
`fact_content_daily_performance` represents a single content item (`content_hash_id`)
belonging to a single client (`client_hash_id`), on a single `report_date`.

**Time window: `month=2026-03`.** This is a mid-panel month, as recommended — the final
month (June 2026) is deliberately excluded from development since it's the natural
outcome window for any past→future label, and using it now would mean testing on the
same future I'd eventually want to predict.

I verify both of these claims with real queries in Section 3 below.

In [ ]:
# Verify: one row = one content page, on one day, for one client
grain_check = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Rows where (date, client, content) is duplicated:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where (date, client, content) is duplicated: 0


,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

Sorting the fields I plan to touch for Lane 2 (Refresh / Content Opportunity Scoring):

**Features** (observable signals, known before any decision point):
- `impressions`, `clicks` — daily GSC signal, safe to use as-is
- `avg_position` — daily search ranking position
- `sessions`, `engagement_rate` — daily GA4 signal (where available)
- `content_age_days` (derived from `dim_content.content_created_at` vs `report_date`)

**Label / proxy:**
- A future-looking decline label I will define myself: whether a page's impressions or
  clicks fall over a forward window relative to a trailing window. I have not built this
  yet — this notebook only proves the raw signals exist and are usable, not the final label.

**Context** (useful for grouping/joins, not used as a feature):
- `client_hash_id`, `content_hash_id`, `report_date` — join keys and grain, not signals
  themselves
- `dim_clients.gsc_data_start`, `ga4_data_start` — needed to know when a client's
  tracking actually began, so I don't mistake "no tracking yet" for "zero traffic"

**Excluded, deliberately:**
- Any


**What this shows:** Of ~9.84M rows in March, only ~414K (about 4.2%) have
`ga4_data_available = True`. Roughly 3.02M rows have a null flag, and ~6.4M are explicitly
False. This means most rows in this slice are search-only (GSC) data — GA4/session-based
features like `sessions` and `engagement_rate` will be usable for only a small fraction of
pages, so any feature relying on them needs a clear "missing vs. zero" distinction, not a
silent fill.

In [ ]:
# Check GA4 availability flag on the March slice
availability_check = con.sql("""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY ga4_data_available
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,row_count
0,<NA>,3018741
1,False,6408671
2,True,413966


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Row count and date span for the March slice
slice_summary = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

slice_summary

,total_rows,min_date,max_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [ ]:
# Build a 5-feature frame, one row per content item, using March data only
feature_frame = con.sql("""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        AVG(gsc_avg_position) AS avg_position_march,
        COUNT(DISTINCT report_date) AS days_with_data,
        SUM(CASE WHEN ga4_data_available THEN ga4_sessions ELSE NULL END) AS sessions_march
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY content_hash_id, client_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 7)


,content_hash_id,client_hash_id,impressions_march,clicks_march,avg_position_march,days_with_data,sessions_march
0,content_67741cce996cfafa,client_62f4a7e64f5e0096,46.0,1.0,4.828125,31,NaN
1,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,899.0,1.0,5.145765,31,NaN
2,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,3108.0,0.0,6.969536,31,NaN
3,content_275b6f7f733016d4,client_62f4a7e64f5e0096,810.0,1.0,4.866123,31,NaN
4,content_4dc944b7d0b65ecc,client_62f4a7e64f5e0096,134.0,0.0,4.627228,31,NaN
5,content_4a1ca0fa5c177e0c,client_62f4a7e64f5e0096,14.0,0.0,4.266667,31,NaN
6,content_92c381fbd361212e,client_62f4a7e64f5e0096,536.0,1.0,4.442543,31,NaN
7,content_97188a7032a705cf,client_62f4a7e64f5e0096,496.0,3.0,4.018509,31,NaN
8,content_c03ecafd4c999f15,client_62f4a7e64f5e0096,10849.0,22.0,8.240351,31,NaN
9,content_e689bc511192751a,client_62f4a7e64f5e0096,61.0,0.0,6.015432,31,NaN


**Five features, each with an "available when?" line:**

1. **`impressions_march`** — knowable at the decision moment because Search Console
   impression counts are logged automatically each day a page appears in search results;
   no future information is needed to sum them by month-end.

2. **`clicks_march`** — knowable at the decision moment for the same reason as impressions:
   GSC logs each click as it happens, so a running total through the end of March is
   available the moment March ends, not after.

3. **`avg_position_march`** — knowable at the decision moment because average search
   position is measured daily from actual SERP placement, not from any outcome that
   happens after the review decision is made.

4. **`days_with_data`** — knowable at the decision moment because it's simply a count of
   how many days in March a page had any recorded activity at all — a fact about the past,
   not a prediction about the future.

5. **`sessions_march`** — knowable at the decision moment where `ga4_data_available` is
   True, because GA4 session counts are logged as they occur. However, this feature is
   missing (NaN) for the ~96% of rows without GA4 tracking, confirmed by the availability
   query in Section 2 — so any model using this feature needs to handle "missing"
   differently from "zero engagement."

**The Leakage Trap**

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Define a simple proxy label: "low performing" page = bottom 25% by clicks this month
feature_frame['low_performer'] = (
    feature_frame['clicks_march'] <= feature_frame['clicks_march'].quantile(0.25)
).astype(int)

# THE TRAP: deliberately add a feature that's derived directly from the label
# This is dishonest on purpose — clicks_per_day is basically clicks_march in disguise
feature_frame['clicks_per_day_LEAKED'] = (
    feature_frame['clicks_march'] / feature_frame['days_with_data'].replace(0, np.nan)
)

honest_features = ['impressions_march', 'avg_position_march', 'days_with_data']
leaked_features = honest_features + ['clicks_per_day_LEAKED']

df_model = feature_frame.dropna(subset=leaked_features + ['low_performer'])

X_honest = df_model[honest_features]
X_leaked = df_model[leaked_features]
y = df_model['low_performer']

Xh_train, Xh_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
Xl_train, Xl_test, _, _ = train_test_split(X_leaked, y, test_size=0.3, random_state=42)

model_honest = LogisticRegression(max_iter=1000).fit(Xh_train, y_train)
model_leaked = LogisticRegression(max_iter=1000).fit(Xl_train, y_train)

auc_honest = roc_auc_score(y_test, model_honest.predict_proba(Xh_test)[:, 1])
auc_leaked = roc_auc_score(y_test, model_leaked.predict_proba(Xl_test)[:, 1])

print(f"Honest features only  → ROC AUC: {auc_honest:.3f}")
print(f"With leaked feature   → ROC AUC: {auc_leaked:.3f}")

Honest features only  → ROC AUC: 0.902
With leaked feature   → ROC AUC: 1.000


**The trap, and what it shows:**

The label `low_performer` is defined directly from `clicks_march` (bottom 25% by clicks).
The "leaked" feature `clicks_per_day_LEAKED` is just `clicks_march` divided by
`days_with_data` — almost a restatement of the label itself, not a genuinely independent
signal.

With honest features only (impressions, avg position, days with data), the model scores
**ROC AUC = 0.901** — strong, and believable for a real-world ranking problem.

The moment `clicks_per_day_LEAKED` is added, the score jumps to **ROC AUC = 1.000** — a
perfect score. This is not a sign of a great model; it's a sign that the model was handed
the answer disguised as a feature. A perfect score on a real-world messy dataset like this
should be treated as a red flag, not a win.

**Fix:** the leaked column is dropped below, and the honest model (0.901 AUC, built only
from features that are genuinely independent of the label) is the one I'm keeping as the
real result.

In [ ]:
# Remove the leaked column — keep only the honest, leakage-free feature frame
feature_frame = feature_frame.drop(columns=['clicks_per_day_LEAKED'])

print("Leaked column removed. Final feature frame columns:")
print(feature_frame.columns.tolist())
print("\nFinal honest model score kept as the real result: ROC AUC = 0.901")

Leaked column removed. Final feature frame columns:
['content_hash_id', 'client_hash_id', 'impressions_march', 'clicks_march', 'avg_position_march', 'days_with_data', 'sessions_march', 'low_performer']

Final honest model score kept as the real result: ROC AUC = 0.901


## 4. Data limits

**Named limitation: GA4 (session/engagement) data is available for only a small fraction
of rows in this slice.** The availability check in Section 2 showed that out of 9,841,378
rows in March 2026, only 413,966 (about 4.2%) have `ga4_data_available = True`. This means
any feature built from sessions or engagement rate — including `sessions_march` in my own
5-feature frame — is missing for roughly 96% of pages, not because those pages have zero
engagement, but because GA4 simply isn't tracked for them yet.

This has a real consequence for Lane 2: a scoring model that leans on engagement signals
will only be meaningfully informed for a small minority of pages, and any comparison
between "engagement-rich" and "engagement-missing" pages needs to explicitly separate
"no GA4 tracking" from "tracked but zero traffic" — treating the two as the same would
quietly bias the model toward whichever clients happen to have GA4 connected, rather than
reflecting real content performance.

This is consistent with what the lane guide itself flags: rows before a client's GA4 start
date contain search data only (`ga4_data_available = FALSE`), and each client's tracking
history begins at a different point — the panel is unbalanced by design, not by error.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.